# MGMT298D: Science and Strategy of AI
### Week 8A - Generative AI APIs
### Application: Building with Language Models

## Setup and API Configuration

In [ ]:
!pip install -q -U google-generativeai

import google.generativeai as genai
from google.colab import userdata
import time

# Store your API key in Colab Secrets:
# 1. Click the 'Key' icon in the left sidebar
# 2. Add new secret named 'GEMINI_API_KEY'
# 3. Paste your API key from https://aistudio.google.com/

api_key = userdata.get('GEMINI_API_KEY')
genai.configure(api_key=api_key)
model = genai.GenerativeModel('gemini-1.5-flash-latest')

# Test the connection
response = model.generate_content("Say 'API connected!' in exactly 2 words.")
print(response.text)

## How LLMs Generate Text: Token Prediction

Large language models work by predicting the next token (word/subword) based on probability distributions. At each step, the model outputs probabilities for all possible next tokens.

In [ ]:
# Conceptual illustration of token prediction
import numpy as np
import matplotlib.pyplot as plt

# Simulated probability distribution for next token after "The cat sat on the"
tokens = ['mat', 'floor', 'chair', 'table', 'roof', 'bed', 'couch', 'other']
probs = [0.35, 0.20, 0.15, 0.12, 0.08, 0.05, 0.03, 0.02]

plt.figure(figsize=(10, 5))
bars = plt.bar(tokens, probs, color='steelblue')
bars[0].set_color('darkgreen')  # Highlight most likely
plt.ylabel('Probability')
plt.xlabel('Possible Next Token')
plt.title('LLM Output: Probability Distribution for Next Token\n"The cat sat on the ___"')
plt.ylim(0, 0.5)
for i, (t, p) in enumerate(zip(tokens, probs)):
    plt.text(i, p + 0.01, f'{p:.0%}', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

print("The model doesn't 'know' the answer - it predicts the most likely next token.")

## Temperature: Controlling Randomness

Temperature controls how "creative" vs "focused" the output is:
- **Low temperature (0.1-0.3)**: More deterministic, picks high-probability tokens
- **High temperature (1.0+)**: More random, spreads probability across tokens

In [ ]:
# Visualize how temperature affects the probability distribution
def apply_temperature(probs, temp):
    """Apply temperature scaling to probabilities."""
    log_probs = np.log(probs)
    scaled = log_probs / temp
    exp_scaled = np.exp(scaled)
    return exp_scaled / exp_scaled.sum()

base_probs = np.array([0.35, 0.20, 0.15, 0.12, 0.08, 0.05, 0.03, 0.02])
tokens = ['mat', 'floor', 'chair', 'table', 'roof', 'bed', 'couch', 'other']

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
temps = [0.3, 1.0, 2.0]
titles = ['Low Temp (0.3)\nMore Focused', 'Normal Temp (1.0)\nBalanced', 'High Temp (2.0)\nMore Creative']

for ax, temp, title in zip(axes, temps, titles):
    adjusted = apply_temperature(base_probs, temp)
    ax.bar(tokens, adjusted, color='steelblue')
    ax.set_title(title)
    ax.set_ylim(0, 0.8)
    ax.set_xticklabels(tokens, rotation=45, ha='right')
    ax.set_ylabel('Probability')

plt.tight_layout()
plt.show()

In [ ]:
# See temperature in action with a real model
prompt = "Write a one-sentence company slogan for a coffee shop."

print("=== Temperature 0.2 (Focused) ===")
for i in range(3):
    response = model.generate_content(prompt, generation_config={'temperature': 0.2})
    print(f"{i+1}. {response.text.strip()}")

print("\n=== Temperature 1.5 (Creative) ===")
for i in range(3):
    response = model.generate_content(prompt, generation_config={'temperature': 1.5})
    print(f"{i+1}. {response.text.strip()}")

## Top-K and Top-P Sampling

Beyond temperature, we can control which tokens are even considered:
- **Top-K**: Only consider the K most likely tokens
- **Top-P (nucleus)**: Only consider tokens until cumulative probability reaches P

In [ ]:
# Visualize Top-K and Top-P
probs = np.array([0.35, 0.20, 0.15, 0.12, 0.08, 0.05, 0.03, 0.02])
tokens = ['mat', 'floor', 'chair', 'table', 'roof', 'bed', 'couch', 'other']
cumsum = np.cumsum(probs)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Top-K visualization
k = 3
colors = ['darkgreen' if i < k else 'lightgray' for i in range(len(tokens))]
ax1.bar(tokens, probs, color=colors)
ax1.set_title(f'Top-K Sampling (K={k})\nOnly consider top {k} tokens')
ax1.set_ylabel('Probability')
ax1.set_xticklabels(tokens, rotation=45, ha='right')

# Top-P visualization
p = 0.7
colors = ['darkgreen' if cumsum[i] <= p or i == 0 else 'lightgray' for i in range(len(tokens))]
ax2.bar(tokens, probs, color=colors)
ax2.axhline(y=0, color='red', linestyle='--', alpha=0.5)
ax2.set_title(f'Top-P Sampling (P={p})\nTokens until cumulative prob reaches {p:.0%}')
ax2.set_ylabel('Probability')
ax2.set_xticklabels(tokens, rotation=45, ha='right')

plt.tight_layout()
plt.show()

print(f"Cumulative probabilities: {[f'{c:.0%}' for c in cumsum]}")

In [ ]:
# Compare Top-K settings
prompt = "List 5 creative names for a tech startup:"

print("=== Top-K = 10 (Fewer choices, more predictable) ===")
response = model.generate_content(prompt, generation_config={'top_k': 10, 'temperature': 1.0})
print(response.text)

print("\n=== Top-K = 100 (More choices, more variety) ===")
response = model.generate_content(prompt, generation_config={'top_k': 100, 'temperature': 1.0})
print(response.text)

## Building a Reusable API Function

In [ ]:
def generate(prompt, temperature=1.0, top_k=40, top_p=0.95, max_tokens=500):
    """Reusable function to generate text with configurable parameters."""
    try:
        response = model.generate_content(
            prompt,
            generation_config={
                'temperature': temperature,
                'top_k': top_k,
                'top_p': top_p,
                'max_output_tokens': max_tokens
            }
        )
        return response.text
    except Exception as e:
        return f"Error: {e}"

# Test the function
print(generate("Explain machine learning in one sentence.", temperature=0.3))

## Application: Structured Output Generation

APIs can generate structured data like JSON, which is useful for building applications.

In [ ]:
import json

prompt = """
Analyze this customer review and return JSON:

Review: "The laptop is fast and the screen is beautiful, but the battery only lasts 3 hours. 
Customer support was helpful when I called about it."

Return ONLY valid JSON in this format:
{
  "sentiment": "positive/negative/mixed",
  "pros": ["list of positives"],
  "cons": ["list of negatives"],
  "summary": "one sentence summary"
}
"""

response = generate(prompt, temperature=0.3)  # Low temp for consistent formatting
print("Raw response:")
print(response)

# Parse the JSON
try:
    # Clean up response (remove markdown code blocks if present)
    clean_response = response.replace('```json', '').replace('```', '').strip()
    data = json.loads(clean_response)
    print("\nParsed JSON:")
    print(f"Sentiment: {data['sentiment']}")
    print(f"Pros: {data['pros']}")
    print(f"Cons: {data['cons']}")
except json.JSONDecodeError as e:
    print(f"Failed to parse JSON: {e}")

## Application: Batch Processing with the API

In [ ]:
# Process multiple items through the API
products = [
    "Wireless noise-canceling headphones",
    "Smart home security camera",
    "Portable laptop charger"
]

results = []
for product in products:
    prompt = f"Write a 2-sentence product description for: {product}"
    description = generate(prompt, temperature=0.7)
    results.append({"product": product, "description": description.strip()})
    time.sleep(0.5)  # Rate limiting

print("=== Generated Product Descriptions ===")
for r in results:
    print(f"\n{r['product']}:")
    print(r['description'])

## API Best Practices

In [ ]:
def robust_generate(prompt, max_retries=3, **kwargs):
    """Production-ready API call with error handling and retries."""
    for attempt in range(max_retries):
        try:
            response = model.generate_content(
                prompt,
                generation_config=kwargs
            )
            return {"success": True, "text": response.text, "attempt": attempt + 1}
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)  # Exponential backoff
            else:
                return {"success": False, "error": str(e), "attempt": attempt + 1}

# Test the robust function
result = robust_generate("What is 2+2?", temperature=0.1)
print(f"Success: {result['success']}")
print(f"Response: {result.get('text', result.get('error'))}")
print(f"Attempts: {result['attempt']}")

In [ ]:
# Summary of generation parameters
summary = """
=== KEY API PARAMETERS ===

TEMPERATURE (0.0 - 2.0)
  - Low (0.1-0.3): Deterministic, consistent outputs
  - Medium (0.7-1.0): Balanced creativity
  - High (1.2+): More random, creative outputs
  
TOP-K (1 - vocab size)
  - Lower K: Only consider most likely tokens
  - Higher K: Consider more token options
  
TOP-P (0.0 - 1.0)
  - Lower P: Nucleus of highest probability tokens
  - Higher P: Include more diverse tokens
  
MAX_TOKENS
  - Controls maximum response length
  - Affects cost (pay per token)

=== USE CASE RECOMMENDATIONS ===
- Factual Q&A: temp=0.1-0.3, top_k=10
- Creative writing: temp=1.0-1.5, top_k=50
- Code generation: temp=0.2, top_p=0.9
- Structured output: temp=0.1-0.3 (for consistency)
"""
print(summary)